In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Đường dẫn data 
root = Path.cwd()
data_dir = root / "data" / "raw"

if not data_dir.exists():
    data_dir = root.parent / "data" / "raw"

# Load dữ liệu từ thư mục data/raw
orders = pd.read_csv(
    data_dir / "transaction/orders.csv",
    parse_dates=["order_date"],
    low_memory=False
)

order_items = pd.read_csv(data_dir / "transaction/order_items.csv", low_memory=False)
payments = pd.read_csv(data_dir / "transaction/payments.csv", low_memory=False)
returns = pd.read_csv(data_dir / "transaction/returns.csv", low_memory=False)

products = pd.read_csv(data_dir / "master/products.csv", low_memory=False)
customers = pd.read_csv(data_dir / "master/customers.csv", low_memory=False)
geography = pd.read_csv(data_dir / "master/geography.csv", low_memory=False)

web_traffic = pd.read_csv(data_dir / "operational/web_traffic.csv", low_memory=False)

sales = pd.read_csv(
    data_dir / "analytical/sales.csv",
    parse_dates=["Date"],
    low_memory=False
)

# Kiểm tra nhanh dữ liệu đã load
print("Loaded tables:")
for name, df in {
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "returns": returns,
    "products": products,
    "customers": customers,
    "geography": geography,
    "web_traffic": web_traffic,
    "sales": sales,
}.items():
    print(f"{name}: {df.shape}")

Loaded tables:
orders: (646945, 8)
order_items: (714669, 7)
payments: (646945, 4)
returns: (39939, 7)
products: (2412, 8)
customers: (121930, 7)
geography: (39948, 4)
web_traffic: (3652, 7)
sales: (3833, 3)


In [ ]:
# Q1: khoảng cách ngày giữa các lần mua (lấy median)

df = orders.sort_values(["customer_id", "order_date"])

# Tính số ngày giữa 2 đơn liên tiếp của mỗi khách
df["gap_days"] = df.groupby("customer_id")["order_date"].diff().dt.days

# Bỏ các giá trị NaN (những khách chỉ có 1 đơn sẽ không có gap)
median_gap = df["gap_days"].dropna().median()

print("Q1 - Median gap (days):", median_gap)

#Median gap (days):144.0-->C

Q1 - Median gap (days): 144.0


In [ ]:
# Q2: segment có tỷ suất lợi nhuận trung bình cao nhất

# Tính margin cho từng sản phẩm
products["margin"] = (products["price"] - products["cogs"]) / products["price"]

# Lấy trung bình theo segment
segment_margin = (
    products.groupby("segment")["margin"]
    .mean()
    .sort_values(ascending=False)
)

print("\nQ2 - Segment có margin cao nhất:")
print(segment_margin.head(1))

print("\nTop các segment:")
print(segment_margin.head())

#Segment có margin cao nhất: Standard-->D


Q2 - Segment có margin cao nhất:
segment
Standard    0.313442
Name: margin, dtype: float64

Top các segment:
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Name: margin, dtype: float64


In [ ]:
# Q3: lý do trả hàng phổ biến nhất (Streetwear)

df = returns.merge(products[["product_id", "category"]], on="product_id", how="left")

# Lọc sản phẩm thuộc Streetwear
streetwear = df[df["category"] == "Streetwear"]

# Đếm lý do trả hàng
reason_counts = streetwear["return_reason"].value_counts()

print("\nQ3 - Lý do trả hàng nhiều nhất (Streetwear):")
print(reason_counts.head(1))

print("\nTất cả lý do:")
print(reason_counts)

#Lý do trả hàng nhiều nhất (Streetwear): wrong_size-->B


Q3 - Lý do trả hàng nhiều nhất (Streetwear):
return_reason
wrong_size    7626
Name: count, dtype: int64

Tất cả lý do:
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64


In [ ]:
# Q4: nguồn traffic có bounce rate trung bình thấp nhất

# Tính trung bình bounce_rate theo từng nguồn
source_bounce = (
    web_traffic.groupby("traffic_source")["bounce_rate"]
    .mean()
    .sort_values()
)

print("\nQ4 - Nguồn có bounce rate thấp nhất:")
print(source_bounce.head(1))

print("\nTất cả nguồn:")
print(source_bounce)
# Nguồn có bounce rate thấp nhất: email_campaign-->C


Q4 - Nguồn có bounce rate thấp nhất:
traffic_source
email_campaign    0.004458
Name: bounce_rate, dtype: float64

Tất cả nguồn:
traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64


In [ ]:
# Q5: % dòng có áp dụng khuyến mãi (promo_id không null)

total_rows = len(order_items)

# Check cột promo_id
promo_rows = order_items["promo_id"].notna().sum()

percent = (promo_rows / total_rows) * 100

print("\nQ5 - Số dòng có promo:", promo_rows)
print(f"Q5 - Tỷ lệ: {percent:.2f}%")

#Tỷ lệ: 38.66%-->C


Q5 - Số dòng có promo: 276316
Q5 - Tỷ lệ: 38.66%


In [ ]:
# Q6: nhóm tuổi có số đơn trung bình / khách cao nhất

df = customers[["customer_id", "age_group"]].dropna()
df = df.merge(orders[["customer_id", "order_id"]], on="customer_id", how="left")

# Tính tổng đơn và số khách theo từng nhóm tuổi
stats = df.groupby("age_group").agg(
    total_orders=("order_id", "count"),
    total_customers=("customer_id", "nunique")
)

# Tính trung bình
stats["avg_orders"] = stats["total_orders"] / stats["total_customers"]

# Sắp xếp giảm dần
stats = stats.sort_values("avg_orders", ascending=False)

print("\nQ6 - Nhóm tuổi mua nhiều nhất:")
print(stats.head(1))

print("\nTất cả nhóm tuổi:")
print(stats)

#Nhóm tuổi mua nhiều nhất: 55+-->A




Q6 - Nhóm tuổi mua nhiều nhất:
           total_orders  total_customers  avg_orders
age_group                                           
55+               72760            13457    5.406851

Tất cả nhóm tuổi:
           total_orders  total_customers  avg_orders
age_group                                           
55+               72760            13457    5.406851
45-54            124138            23172    5.357241
35-44            170368            31920    5.337343
25-34            190622            36342    5.245226
18-24             89057            17039    5.226656


In [ ]:
# Q7: region có tổng doanh thu cao nhất

df = orders.merge(geography[["zip", "region"]], on="zip", how="left")
df = df.merge(order_items[["order_id", "unit_price", "quantity"]], on="order_id", how="left")

# Tính doanh thu
df["revenue"] = df["unit_price"] * df["quantity"]

# Tổng doanh thu theo region
region_rev = (
    df.groupby("region")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

print("\nQ7 - Region có doanh thu cao nhất:")
print(region_rev.head(1))

print("\nTất cả region:")
print(region_rev)

#Region có doanh thu cao nhất: East-->C


Q7 - Region có doanh thu cao nhất:
region
East    7.637533e+09
Name: revenue, dtype: float64

Tất cả region:
region
East       7.637533e+09
Central    4.941908e+09
West       3.851035e+09
Name: revenue, dtype: float64


In [ ]:
# Q8: phương thức thanh toán phổ biến nhất trong các đơn bị huỷ

df = orders[orders["order_status"] == "cancelled"]

# Đếm số lần xuất hiện của từng payment_method
payment_counts = df["payment_method"].value_counts()

print("\nQ8 - Payment method nhiều nhất (cancelled):")
print(payment_counts.head(1))

print("\nTất cả phương thức:")
print(payment_counts)

#Payment method nhiều nhất (cancelled): credit_card-->A



Q8 - Payment method nhiều nhất (cancelled):
payment_method
credit_card    28452
Name: count, dtype: int64

Tất cả phương thức:
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64


In [ ]:
# Q9: size có tỷ lệ trả hàng cao nhất

ret = returns.merge(products[["product_id", "size"]], on="product_id", how="left")
items = order_items.merge(products[["product_id", "size"]], on="product_id", how="left")

# Đếm số lượng theo size
ret_count = ret["size"].value_counts()
item_count = items["size"].value_counts()

# Tính return rate
rate = (ret_count / item_count).sort_values(ascending=False)

print("\nQ9 - Size có return rate cao nhất:")
print(rate.head(1))

print("\nReturn rate theo size:")
print(rate)

#Size có return rate cao nhất: S-->A


Q9 - Size có return rate cao nhất:
size
S    0.056515
Name: count, dtype: float64

Return rate theo size:
size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
Name: count, dtype: float64


In [ ]:
# Q10: số kỳ trả góp có giá trị thanh toán trung bình cao nhất

# Tính giá trị trung bình theo số kỳ
installment_avg = (
    payments.groupby("installments")["payment_value"]
    .mean()
    .sort_values(ascending=False)
)

print("\nQ10 - Installments cao nhất:")
print(installment_avg.head(1))

print("\nTất cả installments:")
print(installment_avg)

#Q10 - Installments cao nhất: 6-->C


Q10 - Installments cao nhất:
installments
6    24446.654403
Name: payment_value, dtype: float64

Tất cả installments:
installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64


In [21]:
# Tổng hợp đáp án

answers = {
    "Q1_median_inter_order_gap_days": float(q1_median_gap),
    "Q2_best_segment": q2_by_segment.idxmax(),
    "Q3_top_streetwear_return_reason": q3_reason_counts.idxmax(),
    "Q4_lowest_bounce_source": q4_by_source.idxmin(),
    "Q5_promo_usage_percent": float(q5_ratio_percent),
    "Q6_best_age_group": q6_stats["avg_orders_per_customer"].idxmax(),
    "Q7_top_region_estimated_revenue": q7_result.idxmax(),
    "Q8_top_cancelled_payment_method": q8_payment_counts.idxmax(),
    "Q9_highest_return_rate_size": q9_return_rate.idxmax(),
    "Q10_best_installments": int(q10_by_installments.idxmax()),
}

print("\n--- TỔNG HỢP KẾT QUẢ ---")
print(pd.Series(answers))


--- TỔNG HỢP KẾT QUẢ ---
Q1_median_inter_order_gap_days              144.0
Q2_best_segment                          Standard
Q3_top_streetwear_return_reason        wrong_size
Q4_lowest_bounce_source            email_campaign
Q5_promo_usage_percent                  38.663493
Q6_best_age_group                             55+
Q7_top_region_estimated_revenue              East
Q8_top_cancelled_payment_method       credit_card
Q9_highest_return_rate_size                     S
Q10_best_installments                           6
dtype: object
